# 18 — PCA Baseline (dimensionality reduction, no WJ awareness)

**Purpose:** Compare against learned intersection-min MLP (nb 16).  
PCA is a standard linear dimensionality reduction with no notion of Weighted Jaccard similarity.

**Pipeline:**
- Fit PCA on corpus: 18499-D → 512-D (linear, variance-maximizing)
- Stage-1 search: cosine similarity on 512-D PCA embeddings (PCA components can be negative; intersection does not apply)
- Stage-2 (optional): exact raw WJ ratio rerank on original 18k vectors — identical to nb 16

**Expected outcome:** PCA stage-1 recall << MLP stage-1 recall, showing that linear compression without WJ training signal fails to preserve shape similarity structure.

**Prereqs:** `/tmp/qt_10k.npy`, `/tmp/gt_lookup_10k.pkl` from `00_cache_data.ipynb`

In [1]:
# ── Configuration ────────────────────────────────────────────────────────────
dataset_name   = "10k"
device_str     = "cuda:0"
n_components   = 512

QUERY_START_10K  = 8000
QUERY_START_FULL = 187019

candidate_ks        = [500, 1000]
rerank_batch_size   = 16
search_query_chunk  = 64     # queries per GPU batch for cosine search
search_corpus_chunk = 8000   # corpus chunk for GPU cosine search

out_path = "/tmp/results_pca_baseline.pkl"
seed = 42

In [2]:
import gc
import pickle
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.decomposition import PCA
from tqdm import tqdm

device = torch.device(device_str if torch.cuda.is_available() else "cpu")
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

print(f"device={device} | n_components={n_components} | dataset={dataset_name}")

device=cuda:0 | n_components=512 | dataset=10k


In [3]:
# ── Load data ────────────────────────────────────────────────────────────────
if dataset_name == "10k":
    qt = np.load("/tmp/qt_10k.npy")
    with open("/tmp/gt_lookup_10k.pkl", "rb") as f:
        gt = pickle.load(f)
    query_start = QUERY_START_10K
elif dataset_name == "full":
    qt = np.load("/tmp/qtree_vectors_full.npy")
    with open("/tmp/gt_lookup_full.pkl", "rb") as f:
        gt = pickle.load(f)
    query_start = QUERY_START_FULL
else:
    raise ValueError(dataset_name)

corpus_qt   = qt[:query_start]
query_qt    = qt[query_start:]
corpus_sums = corpus_qt.sum(axis=1)

print(f"qt={qt.shape} | corpus={corpus_qt.shape} | queries={query_qt.shape}")

qt=(10000, 18499) | corpus=(8000, 18499) | queries=(2000, 18499)


In [4]:
# ── Fit PCA on corpus only (no data leakage from queries) ───────────────────
print(f"Fitting PCA({n_components}) on corpus {corpus_qt.shape} ...")
t0 = time.time()
pca = PCA(n_components=n_components, random_state=seed)
pca.fit(corpus_qt)
explained = pca.explained_variance_ratio_.sum()
print(f"Done in {time.time()-t0:.1f}s | explained variance: {explained:.4f} ({explained*100:.1f}%)")

# Project all vectors
corpus_embs = pca.transform(corpus_qt).astype(np.float32)
query_embs  = pca.transform(query_qt).astype(np.float32)
print(f"corpus_embs={corpus_embs.shape} | query_embs={query_embs.shape}")
print(f"Value range — corpus: [{corpus_embs.min():.3f}, {corpus_embs.max():.3f}]")

Fitting PCA(512) on corpus (8000, 18499) ...
Done in 5.6s | explained variance: 1.0000 (100.0%)
corpus_embs=(8000, 512) | query_embs=(2000, 512)
Value range — corpus: [-0.001, 0.010]


In [5]:
# ── Eval helpers (same as nb 16) ─────────────────────────────────────────────
def recall_at_k(gt_lookup, nbrs, query_start_id, k):
    total, count = 0.0, 0
    for i, ids in enumerate(nbrs):
        qid    = query_start_id + i
        gt_set = set(gt_lookup.get(qid, [])[:k])
        if not gt_set:
            continue
        total += len(gt_set & set(ids[:k])) / len(gt_set)
        count += 1
    return total / count if count else 0.0


def eval_recall_dict(gt_lookup, nbrs, query_start_id, max_k):
    return {
        k: recall_at_k(gt_lookup, nbrs, query_start_id, k)
        for k in (10, 50, 100, 500)
        if k <= max_k
    }


@torch.no_grad()
def knn_cosine_gpu(query_embs, corpus_embs, k, dev,
                   query_chunk=64, corpus_chunk=8000):
    """Top-k by cosine similarity — appropriate for PCA embeddings (can be negative)."""
    # L2-normalize for cosine via dot product
    q_t = torch.from_numpy(query_embs).to(dev, dtype=torch.float32)
    c_t = torch.from_numpy(corpus_embs).to(dev, dtype=torch.float32)
    q_t = F.normalize(q_t, dim=1)
    c_t = F.normalize(c_t, dim=1)

    n_q, n_c = q_t.shape[0], c_t.shape[0]
    top_ids    = np.zeros((n_q, k), dtype=np.int64)
    top_scores = np.full((n_q, k), -2.0, dtype=np.float32)

    for qs in tqdm(range(0, n_q, query_chunk), desc="KNN cosine"):
        qe  = min(qs + query_chunk, n_q)
        qb  = q_t[qs:qe]
        best_scores = torch.full((qb.shape[0], k), -2.0, device=dev)
        best_ids    = torch.zeros((qb.shape[0], k), dtype=torch.long, device=dev)

        for cs in range(0, n_c, corpus_chunk):
            ce      = min(cs + corpus_chunk, n_c)
            cb      = c_t[cs:ce]
            scores  = qb @ cb.T                          # (Bq, Bc)
            cand_ids = torch.arange(cs, ce, device=dev).expand(qb.shape[0], -1)
            merged_scores = torch.cat([best_scores, scores], dim=1)
            merged_ids    = torch.cat([best_ids, cand_ids], dim=1)
            new_scores, order = torch.topk(
                merged_scores, k=min(k, merged_scores.shape[1]), dim=1
            )
            best_ids    = torch.gather(merged_ids, 1, order)
            best_scores = new_scores

        top_ids[qs:qe]    = best_ids.cpu().numpy()
        top_scores[qs:qe] = best_scores.cpu().numpy()

    return [(top_ids[i].tolist(), top_scores[i].tolist()) for i in range(n_q)]


def rerank_wj_gpu(query_qt, nbrs_ids, corpus_qt, corpus_sums, dev, batch_size=16):
    """Exact raw WJ ratio rerank on original 18k vectors — identical to nb 16."""
    corpus_t       = torch.from_numpy(corpus_qt).to(device=dev, dtype=torch.float32)
    corpus_sums_t  = torch.from_numpy(corpus_sums).to(device=dev, dtype=torch.float32)
    reranked = []
    for start in tqdm(range(0, len(nbrs_ids), batch_size), desc="Raw WJ ratio rerank"):
        batch  = nbrs_ids[start:start + batch_size]
        groups = {}
        for offset, ids in enumerate(batch):
            ids_arr = np.asarray(ids, dtype=np.int64)
            groups.setdefault(len(ids_arr), []).append((start + offset, ids_arr))
        for cand_len, items in groups.items():
            if cand_len == 0:
                for _ in items:
                    reranked.append([])
                continue
            ids_np   = np.stack([ids for _, ids in items], axis=0)
            query_np = np.stack([query_qt[abs_i] for abs_i, _ in items], axis=0)
            ids_t    = torch.from_numpy(ids_np).to(device=dev)
            q_t      = torch.from_numpy(query_np).to(device=dev, dtype=torch.float32)
            c_t      = corpus_t[ids_t]
            mins     = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
            maxs     = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
            order    = torch.argsort(
                mins / maxs.clamp_min(1e-10), dim=1, descending=True
            ).cpu().numpy()
            for row, (_, ids) in zip(order, items):
                reranked.append(ids[row].tolist())
    return reranked


print("Eval functions defined.")

Eval functions defined.


In [6]:
# ── Run eval ─────────────────────────────────────────────────────────────────
results = {}
max_k   = max(max(candidate_ks), 500)

print("=" * 72)
print("PCA BASELINE — stage-1 cosine on 512-D PCA embeddings")
print("=" * 72)

t0 = time.time()
nbrs_cosine = knn_cosine_gpu(
    query_embs, corpus_embs, k=max_k, dev=device,
    query_chunk=search_query_chunk, corpus_chunk=search_corpus_chunk
)
qps_cosine = len(query_embs) / (time.time() - t0)
ids_only   = [ids for ids, _ in nbrs_cosine]

print(f"\n--- Stage 1: top-{max_k} by COSINE on 512-D PCA ---")
rec = eval_recall_dict(gt, ids_only, query_start, max_k)
results["pca_cosine_no_rerank"] = {**rec, "qps": qps_cosine}
for k, r in rec.items():
    print(f"  R@{k:<4} = {r:.4f}")
print(f"  QPS ≈ {qps_cosine:.1f}")

for k in candidate_ks:
    print(f"\n--- Stage 2: top-{k} PCA cosine candidates + raw WJ RATIO rerank ---")
    cand_ids = [ids[:k] for ids, _ in nbrs_cosine]
    t0 = time.time()
    rr_ids = rerank_wj_gpu(
        query_qt, cand_ids, corpus_qt, corpus_sums, device, rerank_batch_size
    )
    qps = len(query_embs) / (time.time() - t0)
    rec_rr = eval_recall_dict(gt, rr_ids, query_start, k)
    results[f"k{k}_raw_wj_ratio_rerank"] = {**rec_rr, "qps": qps, "k": k}
    for rk, rv in rec_rr.items():
        print(f"  R@{rk:<4} = {rv:.4f}")
    print(f"  QPS ≈ {qps:.1f}")

payload = {
    dataset_name: results,
    "_meta": {
        "method": "PCA",
        "n_components": n_components,
        "stage1_metric": "cosine",
        "rerank": "raw_wj_ratio",
        "explained_variance": float(explained),
        "time": __import__('time').strftime("%Y-%m-%d %H:%M:%S"),
    }
}
with open(out_path, "wb") as f:
    pickle.dump(payload, f)
print(f"\nSaved {out_path}")

print("\n--- Reference: nb 16 intersection-min MLP ---")
print("  Stage-1 R@10 : ~0.673")
print("  K=500 rerank : ~0.996")
print("  K=1000 rerank: ~0.996")

gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

PCA BASELINE — stage-1 cosine on 512-D PCA embeddings


KNN cosine: 100%|██████████| 32/32 [00:00<00:00, 249.59it/s]



--- Stage 1: top-1000 by COSINE on 512-D PCA ---
  R@10   = 0.3808
  R@50   = 0.4811
  R@100  = 0.5413
  R@500  = 0.7556
  QPS ≈ 3541.0

--- Stage 2: top-500 PCA cosine candidates + raw WJ RATIO rerank ---


Raw WJ ratio rerank: 100%|██████████| 125/125 [00:00<00:00, 253.51it/s]


  R@10   = 0.8675
  R@50   = 0.8506
  R@100  = 0.8335
  R@500  = 0.7556
  QPS ≈ 3687.5

--- Stage 2: top-1000 PCA cosine candidates + raw WJ RATIO rerank ---


Raw WJ ratio rerank: 100%|██████████| 125/125 [00:00<00:00, 153.13it/s]


  R@10   = 0.9525
  R@50   = 0.9454
  R@100  = 0.9371
  R@500  = 0.8939
  QPS ≈ 2297.5

Saved /tmp/results_pca_baseline.pkl

--- Reference: nb 16 intersection-min MLP ---
  Stage-1 R@10 : ~0.673
  K=500 rerank : ~0.996
  K=1000 rerank: ~0.996
